In [1]:
# Directory path
dataset="mstro"
directory = f"./../dataset/{dataset}/docs241205/"
directory = f"./../dataset/{dataset}/docs241205/searchable/"
jsonl_file = f"./../dataset/{dataset}/queries.jsonl"

In [ ]:
# from host
docker exec -u root -it <container_id> bash -c "apt update && apt install -y tesseract-ocr"

In [2]:
!pip install pymupdf

In [ ]:
import ocrmypdf
import os

# List only PDF files
pdf_files = [f for f in os.listdir(directory) if f.endswith(".pdf")]
print("PDF files:", pdf_files)


# Define the input and output PDF file paths
for file in pdf_files:
    
    if "searchable" in file:
        continue
    
    input_pdf = directory + file
    output_pdf = input_pdf.replace(".pdf", "_searchable.pdf")  # Replace with your desired output file name
    
    try:
        # Perform OCR on the input PDF and save the searchable version
        ocrmypdf.ocr(input_pdf, output_pdf, deskew=True, force_ocr=True)
        print(f"OCR completed successfully. Searchable PDF saved as '{output_pdf}'")
    except Exception as e:
        print(f"An error occurred during OCR: {e}")


In [ ]:
import os
import fitz  # PyMuPDF
import json

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    extracted_text = []

    for page_num in range(len(doc)):
        text = doc[page_num].get_text("text")  # Extract text
        extracted_text.append(text)
        print(f"Extracted text from page {page_num + 1}")

    return "\n".join(extracted_text)

# Define the input and output PDF file paths
def extract_text(directory, jsonl_file):

    # List only PDF files
    pdf_files = [f for f in os.listdir(directory) if f.endswith(".pdf")]

    print("PDF files:", pdf_files)
    for file in pdf_files:    
        input_pdf = directory + file
        text = extract_text_from_pdf(input_pdf)
        json_entry = {"_id": file, "text": text}
    
        # Append to JSONL file
        with open(jsonl_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(json_entry, ensure_ascii=False) + "\n")
    
        print(f"Processed {input_pdf} and saved to {jsonl_file}")

extract_text(directory, jsonl_file)